# Shared manifold transitions across four CSL methods

This notebook fits one training-owned reference and reuses its held-out event origins for conventional c-GC/c-GC* and fastC-GC/fastC-GC*. Truth labels remain in metadata. All four use native upstream lag support without a local VAR coefficient readout; they constitute one related method family. Execution is opt-in because conventional permutation tests are expensive. Set RUN_COMPARISON=True after inspecting the runtime and settings. Notebook 01 provides the smaller planted-graph sanity check before this manifold comparison.


In [ ]:
import logging
import sys
from importlib.metadata import PackageNotFoundError, version
from pathlib import Path

try:
    _REPOSITORY_ROOT = next(
        path
        for path in (Path.cwd(), *Path.cwd().parents)
        if (path / "notebooks" / "_shared.py").is_file()
    )
except StopIteration as error:
    raise RuntimeError(
        "Could not locate the effectome repository from the notebook working directory"
    ) from error
sys.path[:0] = [
    path
    for path in (str(_REPOSITORY_ROOT / "src"), str(_REPOSITORY_ROOT))
    if path not in sys.path
]

if sys.version_info[:2] != (3, 14):
    raise RuntimeError(
        "Select the project Python 3.14 .venv kernel before running this notebook. "
        f"Current interpreter: {sys.executable}"
    )
_BACKEND_DISTRIBUTIONS = ("fastcgc", "tigramite", "scikit-learn", "kmedoids", "igraph", "leidenalg")
try:
    _BACKEND_VERSIONS = {name: version(name) for name in _BACKEND_DISTRIBUTIONS}
except PackageNotFoundError as error:
    raise RuntimeError(
        "The notebook kernel is missing a required installed backend. Run "
        "uv sync --python 3.14 --locked --extra dev --extra csl --extra community --extra notebooks "
        "from the repository root, then restart the project .venv kernel."
    ) from error
print("Notebook interpreter:", sys.executable)
print("Installed backends:", _BACKEND_VERSIONS)

ROOT = _REPOSITORY_ROOT
logging.basicConfig(level=logging.INFO, format="%(asctime)s %(levelname)s %(message)s")
logging.getLogger("effectome").setLevel(logging.INFO)
_CONVENTIONAL_PYTHON = ROOT / "tools/conventional-cgc/.venv/bin/python"
print("Conventional CPU runtime:", _CONVENTIONAL_PYTHON)
print("Runtime installed:", _CONVENTIONAL_PYTHON.is_file())


In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display

from effectome.experiments.transition_examples import (
    TransitionExampleConfig,
    make_transition_example,
    make_transition_example_windows,
)

print("Project:", ROOT)
print("Python:", sys.version.split()[0])


In [ ]:
import json
from dataclasses import asdict, replace

from effectome.workflows.cross_method import (
    load_cross_method_profile,
    run_cross_method_comparison,
    write_cross_method_report,
)

SCENARIO = "matched_marginals"  # one_to_one | matched_marginals | fixed_circuit | stationary
SEED = 42
RUN_COMPARISON = False
recording = make_transition_example(TransitionExampleConfig(
    scenario=SCENARIO, seed=SEED, n_samples=4096, regime_dwell=512, trial_length=512,
))
windows = make_transition_example_windows(recording, history_length=256, target_length=8, stride=16)
profile, methods = load_cross_method_profile("csl_sanity")
comparison = replace(profile, n_dims=2, n_manifold_states=3, native_states=2, seed=SEED)
selected_names = tuple(methods)
neuron_ids = [str(value) for value in recording.neuron_ids]
groups = {"rotating_pair": neuron_ids[:2]}
output_dir = (
    ROOT / "outputs" / "synthetic-transition-examples" / SCENARIO
    / f"seed-{SEED}" / "four-csl-sanity"
)
print("Scenario:", SCENARIO, "| windows:", windows.n_windows, "| methods:", selected_names)
print("Settings:", comparison)
print("Output directory:", output_dir)


## Fit the shared reference and issue paired predictions

The 256-sample histories support the fast complete-shift threshold beta=0.005. Alpha=0.01 and beta are fixed engineering settings, not calibrated proof of regime recovery. The chronological holdout purges raw history overlap and keeps training-owned scaling, PCA, region definitions, and predictor fitting separate from held-out events. Geometric regions are exploratory summaries. Native unsigned support can supply graph features but does not parameterize a neural forecast or projected dynamical drive. Conventional estimators run in the pinned isolated environment documented in tools/conventional-cgc/README.md.


In [ ]:
report = None
report_path = output_dir / "comparison.json"
if RUN_COMPARISON:
    report = run_cross_method_comparison(
        windows, config=comparison, method_configs=methods,
        neuron_ids=neuron_ids, groups=groups,
    )
elif report_path.is_file():
    report = json.loads(report_path.read_text())
    if set(report["methods"]) != set(methods) or report["config"] != asdict(comparison):
        raise ValueError("Saved report does not match this comparison; rerun with RUN_COMPARISON=True.")
else:
    print("No four-CSL report exists. Inspect settings, then enable RUN_COMPARISON.")

available = {}
if report is not None:
    available = {
        name: result for name, result in report["methods"].items()
        if result["status"] == "available"
    }
    display(pd.DataFrame([
        {
            "method": name,
            "status": result["status"],
            "reason": result.get("reason", ""),
            "log_loss_gain": result.get("prediction", {}).get("log_loss_gain", np.nan),
            "directed": result.get("directed"),
        }
        for name, result in report["methods"].items()
    ]))
    print("Scientific status:", report["scientific_claims"])


## Shared manifold and held-out origins

Every method receives these same positions, frozen region codes, and event rows. PCA is fitted on training histories; coordinates are never fitted independently for each effectome method.


In [ ]:
if report is not None:
    embedding = np.asarray(report["reference"]["embeddings"])
    states = np.asarray(report["reference"]["manifold_states"])
    events = report["events"]
    origins = np.asarray(events["origins"], dtype=int)
    evaluation_rows = np.asarray(events["evaluation_rows"], dtype=int)
    heldout_origins = origins[evaluation_rows]
    fig, axes = plt.subplots(1, 2, figsize=(12, 5), constrained_layout=True)
    axes[0].scatter(*embedding[:, :2].T, c=states, s=15, cmap="viridis", alpha=0.55)
    axes[0].scatter(*embedding[heldout_origins, :2].T, facecolors="none", edgecolors="black", s=50)
    axes[0].set(
        xlabel="Training PCA 1", ylabel="Training PCA 2",
        title="Common regions; black rings = held-out origins",
    )
    times = np.asarray([anchor.anchor_time_seconds for anchor in windows.anchors])
    axes[1].plot(times, states, drawstyle="steps-post")
    axes[1].scatter(times[heldout_origins], states[heldout_origins], marker="x", color="black")
    axes[1].set(
        xlabel="Time (s)", ylabel="Common geometric region",
        title="One state sequence for all methods",
    )
    plt.show()


## Compare predicted transition matrices with one empirical matrix

Matrices average probabilities over identical source-state origins at a fixed horizon. A source state absent from the holdout is masked, not supplied with a fabricated uniform row. Aggregate matrix agreement can hide incorrect event timing, so inspect held-out log loss and calibration scores too.


In [ ]:
if report is not None:
    n_states = comparison.n_manifold_states
    
    def masked_matrix(rows):
        return np.asarray([
            [np.nan] * n_states if row is None else row for row in rows
        ], dtype=float)
    
    panels = [
        ("Observed, shared", report["common_baseline"]["transition_matrices"]["empirical"]),
        ("Activity/history baseline", report["common_baseline"]["transition_matrices"]["baseline_predicted"]),
    ] + [
        (name, result["prediction"]["transition_matrices"]["augmented_predicted"])
        for name, result in available.items()
    ]
    fig, axes = plt.subplots(1, len(panels), figsize=(4 * len(panels), 4), constrained_layout=True)
    for ax, (title, rows) in zip(np.atleast_1d(axes), panels, strict=True):
        image = ax.imshow(np.ma.masked_invalid(masked_matrix(rows)), vmin=0, vmax=1, cmap="viridis")
        ax.set(title=title, xlabel="Future region", ylabel="Current region")
        fig.colorbar(image, ax=ax, shrink=0.65)
    plt.show()
    display(pd.DataFrame([
        {"method": name, **result["prediction"]["augmented_scores"]}
        for name, result in available.items()
    ]))


## Neuron and group feature-omission contributions

Positive omission_delta_log_loss means the omitted graph-feature block helped held-out prediction after conditioning on the baseline and remaining features. Incoming, outgoing, and group blocks can overlap. Correlated neurons may substitute for each other; a joint group score can exceed individual scores. These are predictive feature scores, not physical silencing experiments or proof of biological necessity. All eligible origins in the source state contribute, including stays and competing destinations.


In [ ]:
if report is not None:
    rows = [
        {"method": name, **row}
        for name, result in available.items()
        for row in result["prediction"]["omission_scores"]
    ]
    contributions = pd.DataFrame(rows)
    if contributions.empty:
        print("No available contribution rows; inspect method failure reasons above.")
    else:
        display(contributions.sort_values("omission_delta_log_loss", ascending=False).head(24))
        destination = 1
        source = 0
        subset = contributions[
            (contributions["source_state"] == source) & (contributions["destination_state"] == destination)
        ]
        if not subset.empty:
            values = subset.pivot(index="unit", columns="method", values="omission_delta_log_loss")
            values.plot.barh(figsize=(10, max(3, len(values) * 0.35)))
            plt.axvline(0, color="black", linewidth=0.7)
            plt.xlabel("Omission loss minus full-model loss (nats)")
            plt.title(f"Illustrative common transition {source} → {destination}")
            plt.tight_layout()
            plt.show()


## Save this experimental run

Outputs live below the ignored outputs directory. Run IDs include scenario, seed, and method set. Inspect unavailable methods and raw counts before interpreting a ranking. Use independent seeds and longer histories/dwell combinations to assess robustness; this notebook does not automatically run the publication-scale null calibration.


In [ ]:
if report is not None:
    paths = write_cross_method_report(report, output_dir)
    for path in paths:
        print(path.relative_to(ROOT))
    display(pd.DataFrame(report["family_rank_synthesis"]).head(20))


Try one_to_one, matched_marginals, fixed_circuit and stationary. Compare how much the activity/history baseline already explains. Continue with notebook 03 to distinguish planted correspondences from learned summaries, then notebook 04 for the repeated validation matrix and manuscript exports.
